# C2: the encoder still reproduces its predictions, byte for byte

The default-path check of `tasks/C2-decoder-baseline/task.md`, Part 2. The decoder options went into the code the encoder runs, so this proves they changed nothing: one short DeBERTa run on the code **before** C2 (`main` at `d2f9369`) and one on the C2 branch, same seed and settings, then the two `predictions.jsonl` are compared byte for byte.

One epoch each, one per T4: **~6 minutes**. Run it once, before the experiment, and again only if `src/` changes.

Temporary trial (`scripts/trial_run.py --mode flat`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings:
- **Accelerator:** GPU **T4 x2**
- **Internet:** on (to clone the repo and download the model)
- **Input:** the private `smi-v4-meetings` dataset attached

The repo branch `C2/decoder-baseline` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [4]:
!nvidia-smi

Sun Nov  1 20:59:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Code and packages
Two checkouts of the same repo: `before` is the code as it was, `after` is the C2 branch. `peft` is deliberately **not** installed: the encoder must keep working without it. If Kaggle happens to have it, nothing changes, because the encoder path never imports it.

In [5]:
BEFORE_COMMIT = "d2f9369"  # main , the last commit without any C2 code

!rm -rf /kaggle/working/after /kaggle/working/before
!git clone -b C2/decoder-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/after
!git clone https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/before
!git -C /kaggle/working/before checkout -q {BEFORE_COMMIT}
!git -C /kaggle/working/after log --oneline -3
!git -C /kaggle/working/before log --oneline -1
!pip install -q transformers==5.20.1 sentencepiece==0.2.2

Cloning into '/kaggle/working/after'...
remote: Enumerating objects: 1022, done.
remote: Counting objects: 100% (382/382), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 1022 (delta 201), reused 259 (delta 101), pack-reused 640 (from 1)
Receiving objects: 100% (1022/1022), 866.39 KiB | 6.28 MiB/s, done.
Resolving deltas: 100% (499/499), done.
Cloning into '/kaggle/working/before'...
remote: Enumerating objects: 1022, done.
remote: Counting objects: 100% (382/382), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 1022 (delta 201), reused 259 (delta 101), pack-reused 640 (from 1)
Receiving objects: 100% (1022/1022), 866.39 KiB | 6.51 MiB/s, done.
Resolving deltas: 100% (499/499), done.
8a0d439 (HEAD -> C2/decoder-baseline, origin/C2/decoder-baseline) Merge C2/decoder-baseline: task description
8a57e38 C2: decoder options in the shared model code, tests, Kaggle notebooks
d2f9369 (origin/main, origin/HEAD) Add CODEOWNERS file for repository own

## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`.

In [6]:
import json
import glob, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

# the same per-meeting files the encoder used : scripts/make_meeting_files.py over release v4
print("data version:", json.load(open(hits[0]))["data_version"])
for split, meetings in (("train", 51), ("val", 12)):
    found = len(glob.glob(f"{DATA_ROOT}/{split}/*.jsonl"))
    print(split, "meetings:", found)
    assert found == meetings, f"expected {meetings} meeting files in {split}/ : is this the right dataset?"

source.json found at: ['/kaggle/input/datasets/mohammedadelguest/smii-v4-meetings/v4_meetings/source.json']
DATA_ROOT = /kaggle/input/datasets/mohammedadelguest/smii-v4-meetings/v4_meetings -> ['source.json', 'test', 'train', 'val']
data version: v4
train meetings: 51
val meetings: 12


## 4. Run both, one per T4
The same flags for both, and no decoder flag: this is the default path. lr 1e-5 is the encoder's adopted value; one epoch is enough to touch loading, training, predicting and the harness.

In [7]:
import subprocess, time

MODEL = "microsoft/deberta-v3-base"
os.makedirs("/kaggle/working/logs", exist_ok=True)

processes = {}
for gpu, side in enumerate(("before", "after")):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src", TOKENIZERS_PARALLELISM="false")
    log_path = f"/kaggle/working/logs/{side}.log"
    cmd = ["python", "scripts/trial_run.py", "--mode", "flat", "--seeds", "42", "--lr", "1e-5", "--epochs", "1",
           "--name", "check", "--data-root", DATA_ROOT, "--model", MODEL, "--out-dir", f"/kaggle/working/runs/{side}"]
    processes[side] = (subprocess.Popen(cmd, cwd=f"/kaggle/working/{side}", env=env,
                                        stdout=open(log_path, "w"), stderr=subprocess.STDOUT), log_path)

while any(process.poll() is None for process, _ in processes.values()):
    time.sleep(30)
    for side, (process, log_path) in processes.items():
        lines = open(log_path).read().strip().splitlines()
        print(f"{side}: {lines[-1] if lines else '(no output yet)'}", flush=True)

for side, (process, log_path) in processes.items():
    print(side, "exit code", process.returncode)
    if process.returncode != 0:
        print(open(log_path).read()[-3000:])

before: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
after: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
before: [flat] seed 42: 2466 train batches per epoch , 2466 optimizer steps | fp16 True | fused AdamW True
after:   family encoder | base fp32 | lora None | trainable parameters 184,427,527 of 184,427,527 (100.00 %)
before:   step 250/2466 (epoch 1) | loss 0.6225 | lr 9.99e-06 | grad_norm 4.84 | loss scale 32768 | 0.5 min
after:   step 250/2466 (epoch 1) | loss 0.6225 | lr 9.99e-06 | grad_norm 4.84 | loss scale 32768 | 0.5 min
before:   step 500/2466 (epoch 1) | loss 0.4715 | lr 8.86e-06 | grad_norm 2.58 | loss scale 32768 | 1.1 min
after:   step 500/2466 (epoch 1) | loss 0.4715 | lr 8.86e-06 | grad_norm 2.58 | loss scale 32768 | 1.1 min
before:   step 750/2466 (epoch 1) | loss 0.4400 | lr 7.73e-06 |

## 5. Compare
`predictions.jsonl` must be identical, byte for byte. `settings.json` will differ, and should: the C2 code records more in it, and that never reaches the predictions.

If the check fails, don't conclude yet: first run this notebook with both sides on the C2 branch. If those two differ as well, the difference is the GPU's, not the code's.

In [8]:
import filecmp, hashlib

paths = {side: f"/kaggle/working/runs/{side}/check_seed42/predictions.jsonl" for side in ("before", "after")}
for side, path in paths.items():
    with open(path, "rb") as f:
        print(side, hashlib.sha256(f.read()).hexdigest(), os.path.getsize(path), "bytes")

identical = filecmp.cmp(paths["before"], paths["after"], shallow=False)
print("predictions identical byte for byte:", identical)
assert identical

scores = {side: json.load(open(f"/kaggle/working/runs/{side}/check_seed42/metrics.json"))["primary"]["macro_f1_6"]
          for side in ("before", "after")}
print("val macro_f1_6:", scores)

before 8e39edc09d6f6f841f3eb9af92a5f8eb30e9dd6066e629c4376b9b3d3b9a3d18 4382038 bytes
after 8e39edc09d6f6f841f3eb9af92a5f8eb30e9dd6066e629c4376b9b3d3b9a3d18 4382038 bytes
predictions identical byte for byte: True
val macro_f1_6: {'before': 0.42419892424275707, 'after': 0.42419892424275707}
